# Function 6: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [ ]:
import numpy as np

input_data = np.load('../../data/initial_data/function_6/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.435568, 0.234968, 0.819048, 0.903328, 0.039242],
    [0.259870, 0.046911, 0.220726, 0.903298, 0.022320],
    [0.5, 0.5, 0.5, 0.5, 0.5]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (20, 5)
After: (22, 5)
[[0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]
 [0.24238435 0.84409997 0.5778091  0.67902128 0.50195289]
 [0.72952261 0.7481062  0.67977464 0.35655228 0.67105368]
 [0.77062024 0.11440374 0.04677993 0.64832428 0.27354905]
 [0.6188123  0.33180214 0.18728787 0.75623847 0.3288348 ]
 [0.78495809 0.91068235 0.7081201  0.95922543 0.0049115 ]
 [0.14511079 0.8966846  0.89632223 0.72627154 0.23627199]
 [0.94506907 0.28845905 0.97880576 0.96165559 0.59801594]
 [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]
 [0.75759436 0.35583141 0.0165229  0.4342072  0.11243304]
 [0.5367969  0.30878091 0.41187929 0.38822518 0.5225283 ]
 [0.95773967 0.23566857 0.09914585 0.15680593 0.07131737]
 [0.6293079  0.80348368 0.81140844 0.04561319 0.11062446]
 [0.02173531 0.42808424 0.83593944 0.48948866 0.51108173]
 [0.43934426 0.69892383 0.42682022 0.10947609 0.87788847]
 [0.25890557 0.79367771 0.6421139  0.19667346 0.59310318]
 [0.43216593 0.71561781 0.3418191  0.7049

In [2]:
output_data = np.load('../initial_data/function_6/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    -0.5489519703174699,
    -1.1254611667938943
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (20,)
After: (22,)
[-0.71426495 -1.20995524 -1.67219994 -1.53605771 -0.82923655 -1.24704893
 -1.23378638 -1.69434344 -2.57116963 -1.30911635 -1.14478485 -1.91267714
 -1.62283895 -1.35668211 -2.0184254  -1.70255784 -1.29424696 -0.93575656
 -2.15576776 -1.74688209 -0.54895197 -1.12546117]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5, 0.5, 0.5]])
actual_output = -0.994699779988161

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 6
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 6, dimension d=5
Data shape: (23, 5) (23,)
Current best observed y: -2.5711696316081234
Current best x: [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,x4,x5,y,log_abs_y,rank_min
8,0.125720,0.862725,0.028544,0.246605,0.751206,-2.571170,0.944361,1
18,0.921776,0.931871,0.414876,0.595057,0.735626,-2.155768,0.768147,2
14,0.439344,0.698924,0.426820,0.109476,0.877888,-2.018425,0.702318,3
11,0.957740,0.235669,0.099146,0.156806,0.071317,-1.912677,0.648504,4
19,0.126679,0.291470,0.064528,0.680515,0.892819,-1.746882,0.557833,5
15,0.258906,0.793678,0.642114,0.196673,0.593103,-1.702558,0.532132,6
7,0.945069,0.288459,0.978806,0.961656,0.598016,-1.694343,0.527295,7
2,0.729523,0.748106,0.679775,0.356552,0.671054,-1.672200,0.514140,8
12,0.629308,0.803484,0.811408,0.045613,0.110624,-1.622839,0.484177,9
3,0.770620,0.114404,0.046780,0.648324,0.273549,-1.536058,0.429219,10


## Gaussian process surrogate + expected improvement for minimisation

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

# Standardise inputs and outputs for numerical stability.
X = input_data.copy()
y = output_data.copy()

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, n_restarts_optimizer=20, random_state=0)
gp.fit(X_scaled, y_scaled)

print("Fitted kernel:", gp.kernel_)
print("Best observed y:", y.min())
print("Best observed x:", X[np.argmin(y)])


Fitted kernel: 1.88**2 * Matern(length_scale=[2.61, 4.35, 5.74, 5.74, 6.62], nu=2.5) + WhiteKernel(noise_level=0.0309)
Best observed y: -2.5711696316081234
Best observed x: [0.12572016 0.86272469 0.02854433 0.24660527 0.75120624]


In [6]:
def expected_improvement_min(X_candidates, gp, y_best_scaled, xi=0.01):
    """Expected improvement for minimisation in scaled y-space."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    improvement = y_best_scaled - mu - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

# Random candidate search in [0, 1]^d. Increase n_candidates for a more exhaustive search.
rng = np.random.default_rng(0)
n_candidates = 20000 if d <= 4 else 50000
candidates = rng.random((n_candidates, d))
candidates_scaled = x_scaler.transform(candidates)

y_best_scaled = y_scaled.min()
ei = expected_improvement_min(candidates_scaled, gp, y_best_scaled, xi=0.01)
mu_scaled, std_scaled = gp.predict(candidates_scaled, return_std=True)
mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
std = std_scaled * y_scaler.scale_[0]

results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
results["pred_mean"] = mu
results["pred_std"] = std
results["expected_improvement"] = ei
results = results.sort_values("expected_improvement", ascending=False)

display(results.head(10))

best_next = results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(dtype=float)
print("Suggested next query:", np.round(best_next, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_next)))


,x1,x2,x3,x4,x5,pred_mean,pred_std,expected_improvement
11364,0.091391,0.998761,0.034169,0.000185,0.979640,-2.860604,0.232798,0.626646
33887,0.022390,0.996289,0.069640,0.115216,0.974653,-2.770750,0.240443,0.470735
1283,0.036979,0.960540,0.140514,0.052897,0.971224,-2.741664,0.236655,0.420805
32131,0.076504,0.908246,0.025880,0.124599,0.950617,-2.754347,0.179788,0.408337
33386,0.000204,0.765898,0.002593,0.031387,0.807902,-2.720197,0.228100,0.381202
18434,0.024572,0.781603,0.026854,0.098712,0.994663,-2.718987,0.218018,0.372420
3311,0.125473,0.869633,0.060715,0.040721,0.911969,-2.736508,0.169840,0.371737
20448,0.056550,0.951781,0.012520,0.130273,0.784582,-2.732911,0.169985,0.365558
16689,0.028717,0.948430,0.104891,0.077981,0.845637,-2.719775,0.205872,0.365543
44708,0.013681,0.998622,0.081074,0.055531,0.666545,-2.709831,0.226669,0.364254


Suggested next query: [9.13910e-02 9.98761e-01 3.41690e-02 1.85000e-04 9.79640e-01]
Portal format: x1=0.091391, x2=0.998761, x3=0.034169, x4=0.000185, x5=0.979640


In [7]:
# Optional 2D visualisation only when d=2
if d == 2:
    grid_res = 150
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = x_scaler.transform(grid)
    mu_grid_scaled, std_grid_scaled = gp.predict(grid_scaled, return_std=True)
    mu_grid = y_scaler.inverse_transform(mu_grid_scaled.reshape(-1, 1)).reshape(grid_res, grid_res)
    ei_grid = expected_improvement_min(grid_scaled, gp, y_best_scaled).reshape(grid_res, grid_res)

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, mu_grid, levels=40)
    plt.colorbar(cf, label="GP predicted mean")
    plt.scatter(input_data[:, 0], input_data[:, 1], c=output_data, edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("GP predicted mean")
    plt.legend(); plt.show()

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, ei_grid, levels=40)
    plt.colorbar(cf, label="Expected improvement")
    plt.scatter(input_data[:, 0], input_data[:, 1], c="white", edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Expected improvement for minimisation")
    plt.legend(); plt.show()
